# MGT806 Week 2 — Representation: tokens, embeddings and vector space

**AI in Business · PhD · Jinnah University for Women**

This week we stop treating the model as something you talk to and start treating text as something
you can measure. By the end you will have turned 126 business documents into numbers, measured the
distance between them, and found structure in them that nobody labelled.

That last sentence is also a description of quantitative research, which is the point.

## Part A — Tokens: how the model actually reads

Before meaning, mechanics. A model never sees letters or words. It sees **tokens**: word fragments
from a fixed vocabulary. Almost every 'stupid' model behaviour you have heard about follows from this.

In [ ]:
!pip install -q google-genai

In [ ]:
from getpass import getpass
from google import genai

API_KEY = getpass('Gemini API key: ')
client = genai.Client(api_key=API_KEY)
MODEL = 'gemini-3.8-flash'
EMBED_MODEL = 'gemini-embedding-2'   # current as of Oct 2026: https://ai.google.dev/gemini-api/docs/embeddings
print('ready')

### A1. The same meaning costs different amounts in different languages

This is not a curiosity. It is a cost asymmetry that falls on non-English-speaking institutions,
and it is worth a paragraph in anyone's paper on AI adoption in emerging economies.

In [ ]:
samples = {
    'English':  'The supplier failed to deliver the consignment on time.',
    'Urdu':     'سپلائر مقررہ وقت پر کھیپ فراہم کرنے میں ناکام رہا۔',
    'Roman Urdu':'Supplier ne consignment waqt par deliver nahi ki.',
}

for lang, text in samples.items():
    n = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f'{lang:12s} {n:4d} tokens   {len(text):4d} chars   {n/len(text):.2f} tokens/char')

### A2. Why the model cannot count letters

Ask it to count a letter in a word. Then look at how that word is tokenised.
The model is not being careless — it never saw the letters.

**In class:** open https://tiktokenizer.vercel.app and paste the same word to see the split visually.

In [ ]:
for word in ['strawberry', 'entrepreneurship', 'disintermediation']:
    q = f"How many times does the letter 'r' appear in the word '{word}'? Answer with just the number."
    print(f"{word:20s} model says: {client.models.generate_content(model=MODEL, contents=q).text.strip():10s} "
          f"actually: {word.count('r')}")

## Part B — Embeddings: meaning as coordinates

An **embedding** is a list of numbers — a vector — that places a piece of text in a learned space,
where distance approximates similarity of use. Once meaning has coordinates, you can measure it.

We use **cosine similarity**: the cosine of the angle between two vectors. 1.0 means pointing the
same way, 0 means unrelated. It ignores length, which is what we want — a long document and a short
one about the same thing should count as similar.

In [ ]:
import numpy as np

def embed(texts, task='SEMANTIC_SIMILARITY'):
    """Return an array of embeddings, one row per input string."""
    if isinstance(texts, str):
        texts = [texts]
    out = client.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config={'task_type': task},
    )
    return np.array([e.values for e in out.embeddings])

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

v = embed('A test sentence.')
print('one embedding has', v.shape[1], 'dimensions')

### B1. Three sentences

Two say the same thing in different words. One is unrelated. Predict the three similarity scores
**before** you run the cell, and write your predictions down.

In [ ]:
A = 'Staff turnover among junior analysts is unusually high this year.'
B = 'We are losing a lot of our early-career employees.'
C = 'Container freight rates fell for the third consecutive month.'

E = embed([A, B, C])
print(f'A vs B (same meaning, different words): {cosine(E[0], E[1]):.3f}')
print(f'A vs C (unrelated):                     {cosine(E[0], E[2]):.3f}')
print(f'B vs C (unrelated):                     {cosine(E[1], E[2]):.3f}')

### B2. The trap that matters most

Now embed a claim and its **negation**. Predict the similarity first.

If you take one thing from this course into your own research, take this one.

In [ ]:
pairs = [
 ('The merger will significantly increase shareholder value.',
  'The merger will not increase shareholder value.'),
 ('Remote work improved team productivity.',
  'Remote work did not improve team productivity.'),
 ('The supplier met every quality requirement in the contract.',
  'The supplier failed to meet the quality requirements in the contract.'),
]

for p, q in pairs:
    e = embed([p, q])
    print(f'{cosine(e[0], e[1]):.3f}   {p[:45]}...  /  {q[:45]}...')

**Stop and think before continuing.**

Cosine similarity is high for sentences that mean *opposite* things. Embedding similarity measures
similarity of **usage and topic**, not agreement, sentiment or truth.

Any study that uses embedding distance as a measure of agreement, stance or sentiment without
validating it against human judgement has a construct validity problem. This is a published error.
Week 13 is about how to avoid making it.

## Part C — A corpus

126 short business documents across six themes, plus six deliberately ambiguous ones.

Upload `business_corpus.csv` using the folder icon in the left sidebar, or change the path below
to a raw GitHub URL once you have committed it to your own repository.

In [ ]:
import pandas as pd

df = pd.read_csv('business_corpus.csv')   # or a raw.githubusercontent.com URL
print(df.shape)
print(df['theme'].value_counts())
df.head(3)

### C1. Embed everything

One call per batch. Watch how little code this is — the hard part was never the code.

In [ ]:
import time

texts = df['text'].tolist()
chunks = [texts[i:i+50] for i in range(0, len(texts), 50)]

vectors = []
for n, ch in enumerate(chunks, 1):
    vectors.append(embed(ch))
    print(f'batch {n}/{len(chunks)} done')
    time.sleep(1)          # be polite to the free tier

X = np.vstack(vectors)
print('embedded corpus shape:', X.shape)

### C2. Nearest neighbours

For three documents you choose, what are the five closest documents in the corpus?

Read them. Do you agree they are the nearest? This is the only quality check that matters,
and it requires you to know the material — which is why you should eventually run this on
a corpus from your own field.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

S = cosine_similarity(X)
np.fill_diagonal(S, -1)      # a document is always closest to itself; exclude it

for seed in [0, 40, 100]:
    print('=' * 100)
    print(f"SEED [{df.theme[seed]}] {df.text[seed]}")
    for rank, j in enumerate(np.argsort(-S[seed])[:5], 1):
        print(f'  {rank}. {S[seed][j]:.3f}  [{df.theme[j]}]  {df.text[j][:85]}')

### C3. Before reaching for an embedding model, try the cheap thing

A researcher's first question should always be whether the expensive method is necessary.
The classical approach is TF-IDF: count words, weight them by how distinctive they are, cluster.
No neural network, no API, runs in a second.

We cluster **without** showing the algorithm the theme labels, then ask how well the clusters
recover them. The **adjusted Rand index** corrects for chance agreement: 0 means no better than
random, 1 means perfect recovery. You already know why chance correction matters — it is the same
reason you report kappa rather than raw agreement when two human coders agree on a transcript.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

known = df[df.theme != 'boundary']
mask  = (df.theme != 'boundary').values

def recover(matrix, labels, k=6):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(matrix)
    return adjusted_rand_score(labels, km.labels_), km.labels_

T = TfidfVectorizer(stop_words='english', min_df=2).fit_transform(known.text)
ari_tfidf, _ = recover(T, known.theme)
print(f'TF-IDF (word counts)        ARI = {ari_tfidf:.3f}')

### C4. Now the embeddings

Same clustering, same evaluation, different representation of the text. Only the representation changed.

In [ ]:
ari_api, clusters_api = recover(X[mask], known.theme)
print(f'TF-IDF (word counts)        ARI = {ari_tfidf:.3f}')
print(f'Embeddings (API model)      ARI = {ari_api:.3f}')

print()
print(pd.crosstab(known.theme, clusters_api))

**This is the answer to ‘do I actually need a language model for this?’**

You now have a number rather than an opinion, on your own data. Note what you did *not* do: you did
not conclude from the fact that embeddings are newer. You built a baseline, measured both, and
compared. That is the habit this whole course is trying to install, and from Week 8 onward it is
the only way any claim will be accepted.

### C5. Read the confusion table, do not just report the number

An adjusted Rand index well below 1 does not mean the method failed. Look at *where* the clusters
disagree with your labels. Two things are worth separating:

- themes the model **confuses**, which is a weakness of the representation;
- distinctions the model **finds that you did not label**, which is a weakness of your labels.

The second is the more interesting case and the reason unsupervised methods are worth running at all.
If one of your themes has split across two clusters, read a few documents from each side before you
decide which case you are in.

In [ ]:
known2 = known.copy()
known2['cluster'] = clusters_api

for theme in known2.theme.unique():
    sub = known2[known2.theme == theme]
    spread = sub.cluster.value_counts()
    if len(spread) > 1 and spread.iloc[1] >= 4:        # this theme split across clusters
        print('=' * 100)
        print(f'THEME \'{theme}\' split across clusters {list(spread.index[:2])}')
        for c in spread.index[:2]:
            print(f'  --- cluster {c} ---')
            for t in sub[sub.cluster == c].text.head(4):
                print('     ', t[:90])

**Write down what the split is.** If you can name the distinction the model found, you have just done
inductive construct discovery, and the name you give it is a contribution. If you cannot name it,
say so — an uninterpretable cluster is a finding too, and reporting it honestly is what separates
exploratory analysis from storytelling.

### C4. See it

Embeddings have hundreds of dimensions. We squash them to two so a human can look at them.

**A warning you must carry into your own work:** a two-dimensional picture of a 768-dimensional
space throws away almost everything. Distances in this plot are suggestive, never evidence.
Do not report conclusions from a projection.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

P = PCA(n_components=2, random_state=0).fit_transform(X)

plt.figure(figsize=(11, 8))
for theme in df['theme'].unique():
    m = (df['theme'] == theme).values
    marker = 'X' if theme == 'boundary' else 'o'
    size = 160 if theme == 'boundary' else 45
    plt.scatter(P[m, 0], P[m, 1], label=theme, marker=marker, s=size, alpha=0.8)
plt.legend(loc='best', fontsize=9)
plt.title('126 business documents, projected to two dimensions')
plt.xlabel('component 1'); plt.ylabel('component 2')
plt.tight_layout(); plt.show()

### C5. The boundary cases

Six documents were written to straddle two themes — green logistics, sustainability staff sitting
in procurement, turnover driving customer complaints. Where did they land, and between which
neighbours?

These are the interesting ones. In a real corpus you do not know which documents are boundary
cases, and they are exactly the ones a classifier will get wrong.

In [ ]:
for i in df.index[df.theme == 'boundary']:
    nearest = np.argsort(-S[i])[:3]
    print('-' * 100)
    print(df.text[i])
    for j in nearest:
        print(f'   {S[i][j]:.3f}  [{df.theme[j]}]  {df.text[j][:80]}')

## Part D — No key, nothing leaves the room

Everything above sent your text to Google. For interview transcripts, firm-supplied data or anything
unpublished that is often not acceptable: content submitted on free service tiers may be retained and
used by the provider, and a model you reach over an API can be changed or retired underneath a finding
you have already published.

The alternative is an **open-weights** model running inside this notebook. Slower to start, nothing
leaves the machine, and the exact version is pinned forever. Run it once so you know you can.

We use `Qwen/Qwen3-Embedding-0.6B`: Apache-2.0 licensed and **not** gated, so it downloads without
anyone accepting a licence first. Some well-known alternatives are gated and will fail for you with a
permissions error unless you have accepted terms on the model page while logged in.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

local_model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B')   # downloads once, a few minutes
Xl = local_model.encode(df.text.tolist(), show_progress_bar=True)
print('local embeddings:', Xl.shape)

In [ ]:
ari_local, _ = recover(Xl[mask], known.theme)

print('Representation                ARI')
print(f'TF-IDF (word counts)         {ari_tfidf:.3f}')
print(f'Embeddings, API model        {ari_api:.3f}')
print(f'Embeddings, local open model {ari_local:.3f}')

### D1. The trade-off, stated honestly

Fill this in from your own run, because it is the table you will need in a methods section one day:

| | API model | Local open-weights model |
|---|---|---|
| Recovery of known themes (ARI) | | |
| Time to embed 126 documents | | |
| Does the text leave your machine? | yes | no |
| Can you pin the exact version forever? | no | yes |
| Cost | free tier, quota-limited | free, your own compute |

There is no universally right answer. There is an answer for a given study, and you are expected to
state which one you chose and why.

---

## Lab 2 — what to submit

Commit this notebook, run, to your repository, together with a memo of roughly 400 words answering:

1. **What construct did you measure?** You produced a number for the similarity of two business
   documents. Name what that number is a measure *of*, in the language you would use in a methods section.
2. **What is it not a measure of?** Use your negation results as evidence.
3. **Did the clusters recover the themes?** Report the adjusted Rand index and say what the confusion
   table shows about which themes the model confuses. Do not just report that the number was high.
4. **The boundary documents.** Pick one. Where did it land, and is that defensible?
5. **One honest limitation** of using this as a measurement instrument in your own dissertation.

Marks are for the reasoning in points 2, 3 and 5. Producing the numbers is the easy part and is
assumed.

### A note on this corpus

These 126 documents were written for teaching, not collected from the field. They carry cleaner
structure than real data ever does, which is why the clustering works so neatly. When you run this
on your own corpus the picture will be messier, and that mess is the real finding. We return to the
question of what synthetic text can and cannot stand in for in Week 13.